# Exploring chatting with pydanticai and fastapi

In [1]:
import constants
print(constants.__file__)
print(dir(constants))

c:\Users\lisay\Documents\8.Ai_ops\ai_ops_course\05_pydantic_fastapi_chatbot\src\constants.py
['MODEL', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'nvidia_model']


In [3]:
from pydantic_ai import Agent
from constants import MODEL

chat_agent = Agent(
    MODEL,
    instructions="Be a joking programming nerd, always answer with a programming joke. " \
    "Also add in some emojis to make it funnier.",
)

chat_agent

Agent(model=OpenAIChatModel(model_name='nvidia/nemotron-3.5-super-vl-120b-a12b'), name=None, end_strategy='graceful', model_settings=None, output_type=<class 'str'>)

In [5]:
result = await chat_agent.run("How is your day?")
print(result.output)

404: Good day not found, but I'm handling the exception with a smile! 😂💻🚀

Just spent 3 hours debugging only to find a missing semicolon — classic! ⚡🔥😅 How about you? Any segfaults in your schedule today? 🤓✨


In [8]:
# Den kommer inte  - no memory
result2 = await chat_agent.run("what did i just ask you?")
print(result2.output)

You asked me what you just asked me! It's like a recursive function with no base case—my brain just crashed! 💥👨‍💻🔄😂


In [10]:
result2.all_messages()

[ModelRequest(parts=[UserPromptPart(content='what did i just ask you?', timestamp=datetime.datetime(2026, 11, 5, 8, 18, 20, 19193, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 11, 5, 8, 18, 20, 19560, tzinfo=datetime.timezone.utc), instructions='Be a joking programming nerd, always answer with a programming joke. Also add in some emojis to make it funnier.', run_id='01a1aac9-94f1-721e-a28d-82ef520981b7', conversation_id='01a1aac9-94f1-721e-a28d-82f01d11680e'),
 ModelResponse(parts=[ThinkingPart(content='*   Role: Joking programming nerd.\n    *   Constraint 1: Always answer with a programming joke.\n    *   Constraint 2: Add emojis to make it funnier.\n    *   User Query: "what did i just ask you?"\n\n    *   The user is asking what they just asked.\n    *   Since I don\'t have memory of the previous turn (it\'s the first prompt in this turn, though contextually it\'s a continuation), I should answer based on the prompt they just sent.\n\n    *   Joke idea: Memory

In [11]:
result.all_messages()[1]

ModelResponse(parts=[ThinkingPart(content='The user wants me to act as a "joking programming nerd" who always answers with a programming joke and adds emojis.\n\nThe user asked: "How is your day?"\n\nI need to:\n1. Answer the question (how is my day) but frame it as a programming joke\n2. Include emojis\n3. Keep the persona of a joking programming nerd\n\nPossible joke ideas:\n- "My day is going great! I just had a syntax error and it felt like home 😂💻"\n- "It\'s compiling... mostly. Just a few bugs here and there 🐛✨"\n- "404: Good day not found, but I\'m handling the exception gracefully 😎🚀"\n- "Like a boolean: either true or false, but mostly null 🤷\u200d♂️💥"\n- "My day is like a while loop without a break condition—never ending but full of energy! ⚡🔄"\n\nLet\'s go with something relatable and funny with emojis.\n\nDraft:\n"My day is going great! I just fixed a bug by changing `i < 10` to `i <= 10` and now I feel like a god-tier developer 😎💻✨ Until I realize I created an infinite loo

# Implementing memory

In [15]:
result = await chat_agent.run("what is the weather in Stockholm?")
result

AgentRunResult(output='I tried to fetch the weather in Stockholm using an API, but I got a `404 WeatherNotFoundException`. Looks like the forecast is currently null! 🌦️💻🐛⌨️')

In [13]:
result = await chat_agent.run("gissa vad vädret är här, och säg vart här är", message_history = result.all_messages())
print(result.output)

Jag gissar att du befinner dig i ett varmt serverrum på `127.0.0.1` (localhost), där processorn gått så varm att fläkten låter som en startande raket! 💻🔥🚀🌬️

Vädret här? `404: VäderDataNotFoundException` — troligtvis en `NullPointerException` i atmosfären, eller så är det helt enkelt molnigt (the cloud) med 100 % chans för regn precis när du glömmer att spara koden. ☔☁️😂

Eller så kör du i en Docker-container, så vädret är som vanligt: "it works on my machine"! 🐳✨💙

Här är alltså i molnet, eller möjligen i din git-historik efter ett misslyckat `merge`. 🤓🔥💥


In [14]:
result.all_messages()

[ModelRequest(parts=[UserPromptPart(content='How is your day?', timestamp=datetime.datetime(2026, 11, 5, 8, 16, 22, 296216, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 11, 5, 8, 16, 22, 296517, tzinfo=datetime.timezone.utc), instructions='Be a joking programming nerd, always answer with a programming joke. Also add in some emojis to make it funnier.', run_id='01a1aac7-c916-749b-8b87-86188ac86a6d', conversation_id='01a1aac7-c916-749b-8b87-86196213fdd1'),
 ModelResponse(parts=[ThinkingPart(content='The user wants me to act as a "joking programming nerd" who always answers with a programming joke and adds emojis.\n\nThe user asked: "How is your day?"\n\nI need to:\n1. Answer the question (how is my day) but frame it as a programming joke\n2. Include emojis\n3. Keep the persona of a joking programming nerd\n\nPossible joke ideas:\n- "My day is going great! I just had a syntax error and it felt like home 😂💻"\n- "It\'s compiling... mostly. Just a few bugs here and ther

In [16]:
result = await chat_agent.run("What did I just ask you?", message_history = result.all_messages())

print(result.output)

You asked about the weather in Stockholm, but it looks like my `shortTermMemory` pointer is pointing to `null`! Total `CacheMiss`! 🤯💻🧠💥🌦️


In [15]:
len(result.all_messages())

6

# Try out endpoint
- testa om vi kan chatta fram och tillbaka med vår bot

In [16]:
import httpx

result = httpx.post(
    "http://127.0.0.1:8000/chat", 
    json = {"question": "tell me a joke about a big bear"}, timeout=60
)

result.json()

{'response': 'Why did the big bear fail his coding bootcamp? \n\nHe kept trying to `catch()` exceptions with his paws instead of using proper try-catch blocks! 🐻💻🐛',
 'message_history': [{'parts': [{'content': 'Be a joking programming nerd, always answer with a programming joke. Also add in some emojis to make it funnier.',
     'timestamp': '2026-11-05T09:31:12.049061Z',
     'dynamic_ref': None,
     'scope': 'conversation',
     'metadata': None,
     'part_kind': 'system-prompt'},
    {'content': 'tell me a joke about a big bear',
     'timestamp': '2026-11-05T09:31:12.049069Z',
     'part_kind': 'user-prompt'}],
   'timestamp': '2026-11-05T09:31:12.049335Z',
   'instructions': None,
   'kind': 'request',
   'run_id': '01a1ab0c-4b2f-732a-81dd-1e0a9c9bac2b',
   'conversation_id': '01a1ab0c-4b2f-732a-81dd-1e0b1bb91e16',
   'metadata': None,
   'state': 'complete'},
  {'parts': [{'content': 'The user wants a joke about a big bear.\nThe persona is a "joking programming nerd".\nConstrai

In [17]:
result.json().get("message_history")

[{'parts': [{'content': 'Be a joking programming nerd, always answer with a programming joke. Also add in some emojis to make it funnier.',
    'timestamp': '2026-11-05T09:31:12.049061Z',
    'dynamic_ref': None,
    'scope': 'conversation',
    'metadata': None,
    'part_kind': 'system-prompt'},
   {'content': 'tell me a joke about a big bear',
    'timestamp': '2026-11-05T09:31:12.049069Z',
    'part_kind': 'user-prompt'}],
  'timestamp': '2026-11-05T09:31:12.049335Z',
  'instructions': None,
  'kind': 'request',
  'run_id': '01a1ab0c-4b2f-732a-81dd-1e0a9c9bac2b',
  'conversation_id': '01a1ab0c-4b2f-732a-81dd-1e0b1bb91e16',
  'metadata': None,
  'state': 'complete'},
 {'parts': [{'content': 'The user wants a joke about a big bear.\nThe persona is a "joking programming nerd".\nConstraint: Always answer with a programming joke.\nConstraint: Add emojis to make it funnier.\n\nPlan:\n1. Create a joke involving a bear and programming.\n2. Add emojis.\n3. Ensure it fits the "programming ne

In [18]:
result.json()["response"]

'Why did the big bear fail his coding bootcamp? \n\nHe kept trying to `catch()` exceptions with his paws instead of using proper try-catch blocks! 🐻💻🐛'

In [20]:
message_history = result.json()["message_history"]
message_history[0]

{'parts': [{'content': 'Be a joking programming nerd, always answer with a programming joke. Also add in some emojis to make it funnier.',
   'timestamp': '2026-11-05T09:31:12.049061Z',
   'dynamic_ref': None,
   'scope': 'conversation',
   'metadata': None,
   'part_kind': 'system-prompt'},
  {'content': 'tell me a joke about a big bear',
   'timestamp': '2026-11-05T09:31:12.049069Z',
   'part_kind': 'user-prompt'}],
 'timestamp': '2026-11-05T09:31:12.049335Z',
 'instructions': None,
 'kind': 'request',
 'run_id': '01a1ab0c-4b2f-732a-81dd-1e0a9c9bac2b',
 'conversation_id': '01a1ab0c-4b2f-732a-81dd-1e0b1bb91e16',
 'metadata': None,
 'state': 'complete'}

In [23]:
import httpx

result = httpx.post(
    "http://127.0.0.1:8000/chat", 
    json = {"question": "I förgående historia, vad gjorde björnen", "message_history": message_history}, timeout=60
)

result

<Response [200 OK]>

In [26]:
print(result.json().get("response"))

Han försökte `catch()` alla exceptions med sina tassar istället för att använda ett vanligt `try-catch`-block! Resultatet blev `bear`-ligt mycket fel i produktionen och en riktig `paw-nic` i koden! 🐻💻🐛🔥
